# Build a MoleculeDataset from SMILES

SMILES → conformers → MoleculeDataset zarr, the input format for evaluation and training. MACE features are computed later, inside the model — not here.

**Needs:** a GPU + MACE deps installed, and a SMILES file (one per line). This mirrors `scripts/dataset_creation/load_from_smiles.py`. See [docs/prepare-a-dataset.md](../docs/prepare-a-dataset.md).

In [ ]:
from pathlib import Path

import torch

from remedi.configuration.dataset_config import (
    DatasetConfig,
    DatasetCreationConfig,
    FilterMoleculeStageConfig,
)
from remedi.data_handling.dataset_creation.generators.smiles_list_generator import (
    SmilesMoleculeGenerator,
    open_smiles_file,
)
from remedi.data_handling.dataset_creation.orchestrator import (
    DatasetConstructionOrchestrator,
)
from remedi.data_handling.dataset_creation.pipeline_stages import (
    ConformerGenerationStage,
    CopyDataStage,
    FilterMoleculeStage,
)

SMILES_FILE = Path("my_molecules.smi")  # <-- edit: one SMILES per line
OUT_DIR = Path("./my_dataset")  # <-- edit

In [ ]:
smiles = open_smiles_file(SMILES_FILE)
creation_config = DatasetCreationConfig(path=OUT_DIR, N_structures=len(smiles))

gen = SmilesMoleculeGenerator(smiles, batch_size=500)
pipeline = [
    FilterMoleculeStage(config=FilterMoleculeStageConfig(max_atoms=100)),
    ConformerGenerationStage(dataset_creation_config=creation_config),
    CopyDataStage(dtype=torch.float64),
]
dataset_config = DatasetConfig(
    atom_chunk=450, molecule_chunk=50, contains_smiles=True, tasks=None
)

DatasetConstructionOrchestrator(
    pipeline=pipeline,
    batch_generator=gen,
    construction_config=creation_config,
    dataset_config=dataset_config,
).build_dataset()
print(f"Wrote dataset to {OUT_DIR}")

To attach labels for downstream training, set `tasks=` on `DatasetConfig` to a `TaskSet` and write targets into the dataset's `tasks/` group — see [docs/prepare-a-dataset.md](../docs/prepare-a-dataset.md).